In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata

# Load your TSV files
source1 = pd.read_csv("../dataset/train/train_source1.tsv", sep="\t")
source2 = pd.read_csv("../dataset/train/train_source2.tsv", sep="\t")
source3 = pd.read_csv("../dataset/train/train_source3.tsv", sep="\t")

ground_truth = pd.read_csv("../dataset/train/train_ground_truth.tsv", sep="\t")

print("Source 1:", source1.shape)
print("Source 2:", source2.shape)
print("Source 3:", source3.shape)
print("Ground truth:", ground_truth.shape)

print("\nColumns:")
print(source1.columns)
print(ground_truth.columns)

Source 1: (154205, 4)
Source 2: (161879, 4)
Source 3: (186868, 4)
Ground truth: (254863, 2)

Columns:
Index(['entity_id', 'business_name', 'business_address', 'country'], dtype='object')
Index(['source1_entity_id', 'matched_entity_ids'], dtype='object')


# Step 1: Normalize efficiently

In [7]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)

    # Lowercase
    text = text.lower()

    # Remove accents
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    # Replace punctuation with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
for df in [source1, source2, source3]:

    df["name_normalized"] = df["business_name"].apply(normalize_text)

    df["address_normalized"] = df["business_address"].apply(normalize_text)

    df["country_normalized"] = df["country"].apply(normalize_text)

In [10]:
source2_small = source2[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

source3_small = source3[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

source2_small["source"] = "source2"
source3_small["source"] = "source3"

reference = pd.concat(
    [source2_small, source3_small],
    ignore_index=True
)

print(reference.shape)
display(source2_small)
display(source3_small)
display(reference.head(10320219))

(348747, 5)


,entity_id,business_name,business_address,country,source
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2
...,...,...,...,...,...
161874,S2-186372959,Public Apex Care Limited,"NO 88 NEAR KHANDAGIRI TEMPLE, C/O DR PRAFULLA ...",India,source2
161875,S2-486217884,SUPREME CHOICE HOVNANIAN,"6309 AIRPORT FREEWAY, HALTOM CITY, TX",US,source2
161876,S2-214470814,"Sharl N. Ccc, Ph.D.","1023 MEADOWLARK DR, CARTERVILLE, IL",US,source2
161877,S2-76544143,Royal Developers Pvt Ltd.,"8-2-293/82/A/623/A, SY. NO. 120, ROAD NO. 34, ...",India,source2


,entity_id,business_name,business_address,country,source
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,source3
1,S3-859268022,International South Consultants Private Ltd,NaN,India,source3
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,source3
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,source3
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India,source3
...,...,...,...,...,...
186863,S3-879271022,Glyphisyn Residential,"Ohio, 1005 Valley Vista Way, Centerville",US,source3
186864,S3-722666596,Baba éngíneering LLP,NaN,India,source3
186865,S3-498125487,All Legacy Fuels Ltd,"12241 22nd Avenue, Burien, Washington",US,source3
186866,S3-749130480,Smt Mangala International School L.L.P.,"Ahmedabad, GJ, 10, Ahmedabad",India,source3


,entity_id,business_name,business_address,country,source
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2
...,...,...,...,...,...
348742,S3-879271022,Glyphisyn Residential,"Ohio, 1005 Valley Vista Way, Centerville",US,source3
348743,S3-722666596,Baba éngíneering LLP,NaN,India,source3
348744,S3-498125487,All Legacy Fuels Ltd,"12241 22nd Avenue, Burien, Washington",US,source3
348745,S3-749130480,Smt Mangala International School L.L.P.,"Ahmedabad, GJ, 10, Ahmedabad",India,source3


# Step 3: Create an exact-match key

In [11]:
source1["exact_key"] = (
    source1["business_name"] + "|" +
    source1["business_address"] + "|" +
    source1["country"]
)

reference["exact_key"] = (
    reference["business_name"] + "|" +
    reference["business_address"] + "|" +
    reference["country"]
)
print(reference.columns)
display(reference.head(10320219))

Index(['entity_id', 'business_name', 'business_address', 'country', 'source',
       'exact_key'],
      dtype='object')


,entity_id,business_name,business_address,country,source,exact_key
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2,राम मार्केटिंग प्राइवेट लिमिटेड|KH NO. -570/13...
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2,"-- Holloway Peak Inc Seafood|105 ELM ST, MORGA..."
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2,"आदित्य प्रॉपर्टीज एलएलपी|G-3/571, GULMOHAR COL..."
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2,"Summit Inc|GREENSBORO, NC, 19 1/2 STARDUST TRA..."
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2,"Delta Tetlecommunication Inc|914 PIERPONT AVE,..."
...,...,...,...,...,...,...
348742,S3-879271022,Glyphisyn Residential,"Ohio, 1005 Valley Vista Way, Centerville",US,source3,"Glyphisyn Residential|Ohio, 1005 Valley Vista ..."
348743,S3-722666596,Baba éngíneering LLP,NaN,India,source3,NaN
348744,S3-498125487,All Legacy Fuels Ltd,"12241 22nd Avenue, Burien, Washington",US,source3,"All Legacy Fuels Ltd|12241 22nd Avenue, Burien..."
348745,S3-749130480,Smt Mangala International School L.L.P.,"Ahmedabad, GJ, 10, Ahmedabad",India,source3,Smt Mangala International School L.L.P.|Ahmed...


# Step 4: Remove useless duplicate keys

In [13]:
reference_unique = reference.drop_duplicates(
    subset=["exact_key"],
    keep=False
)

print("Reference rows:", len(reference))
print("Unique exact keys:", len(reference_unique))
display(reference_unique.head(9887955))

Reference rows: 348747
Unique exact keys: 337086


,entity_id,business_name,business_address,country,source,exact_key
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,source2,राम मार्केटिंग प्राइवेट लिमिटेड|KH NO. -570/13...
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,source2,"-- Holloway Peak Inc Seafood|105 ELM ST, MORGA..."
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,source2,"आदित्य प्रॉपर्टीज एलएलपी|G-3/571, GULMOHAR COL..."
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,source2,"Summit Inc|GREENSBORO, NC, 19 1/2 STARDUST TRA..."
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,source2,"Delta Tetlecommunication Inc|914 PIERPONT AVE,..."
...,...,...,...,...,...,...
348740,S3-814089003,Downing Pémnnmac LLC,"401 Grace Ave, Virginia Beach, Virginia",US,source3,"Downing Pémnnmac LLC|401 Grace Ave, Virginia B..."
348741,S3-402278699,Montay-Cosmetics,"9277 Neiswander Rd, Asvhille, Ohio",US,source3,"Montay-Cosmetics|9277 Neiswander Rd, Asvhille,..."
348742,S3-879271022,Glyphisyn Residential,"Ohio, 1005 Valley Vista Way, Centerville",US,source3,"Glyphisyn Residential|Ohio, 1005 Valley Vista ..."
348744,S3-498125487,All Legacy Fuels Ltd,"12241 22nd Avenue, Burien, Washington",US,source3,"All Legacy Fuels Ltd|12241 22nd Avenue, Burien..."


# Step 5: Exact matching using Pandas merge

In [14]:
exact_matches = source1[
    ["entity_id", "exact_key"]
].merge(
    reference_unique[
        ["exact_key", "entity_id", "source"]
    ],
    on="exact_key",
    how="left",
    suffixes=("_source1", "_matched")
)
print(exact_matches.shape)

exact_matches[exact_matches["entity_id_matched"].notna()]
exact_matches[exact_matches["entity_id_matched"].isna()]


(154205, 4)


,entity_id_source1,exact_key,entity_id_matched,source
0,S1-925783039,"Orelee's Barbershop|1795 Westchester Drive, Hi...",NaN,NaN
1,S1-773889195,"Prime Money|17560 Ellis Road, Tahlequah, OK|US",NaN,NaN
2,S1-377745466,"B+ Retail Inc|1712 Montebello Avenue, Phoenix,...",NaN,NaN
3,S1-133037285,"Christ Chapel|2100 Cameron Drive, Unit APARTME...",NaN,NaN
4,S1-755362802,"Prabhav Business Center|797, Lake Town Block A...",NaN,NaN
...,...,...,...,...
154200,S1-957795759,"Quill Trading Private Limited|Houseb No.3/19, ...",NaN,NaN
154201,S1-668474082,Sterling Technical Worldwide|107 Faulkner Lane...,NaN,NaN
154202,S1-836707650,"Jenna Wojnar Heritage, LLC|Lindenhurst, 448 Re...",NaN,NaN
154203,S1-201407714,Diversified Creative Products Inc|10 Oregon Ro...,NaN,NaN


In [4]:
ground_truth["matched_entity_ids"] = (
    ground_truth["matched_entity_ids"]
    .fillna("")
    .astype(str)
)
gt = ground_truth.set_index(
    "source1_entity_id"
)["matched_entity_ids"]
exact_matches["ground_truth"] = (
    exact_matches["entity_id_source1"].map(gt)
)
display(
    exact_matches[
        [
            "entity_id_source1",
            "entity_id_matched",
            "ground_truth"
        ]
    ].head(20)
)

NameError: name 'exact_matches' is not defined

# Create a blocking key

In [15]:
source1["block_key"] = (
    source1["country"] + "|" +
    source1["business_name"].str[:4]
)

reference["block_key"] = (
    reference["country"] + "|" +
    reference["business_name"].str[:4]
)

print(source1["block_key"].head())
print(reference["block_key"].head())
block_sizes = reference.groupby("block_key").size()

print("Number of blocks:", len(block_sizes))
print("Average block size:", block_sizes.mean())
print("Median block size:", block_sizes.median())
print("Maximum block size:", block_sizes.max())
print(
    block_sizes
    .sort_values(ascending=False)
    .head(20)
)


0       US|Orel
1       US|Prim
2       US|B+ R
3       US|Chri
4    India|Prab
Name: block_key, dtype: object
0    India|राम 
1       US|-- H
2    India|आदित
3       US|Summ
4       US|Delt
Name: block_key, dtype: object
Number of blocks: 56904
Average block size: 6.128637705609448
Median block size: 1.0
Maximum block size: 2708
block_key
US|The        2708
India|Priv    2275
US|LLC        2053
India|Smt     1876
India|Shri    1869
India|M/s     1847
India|Sri     1814
US|Pedi       1650
US|Inte       1553
US|Inc        1061
India|Limi     948
US|Prim        885
US|Metr        829
US|Fami        750
India|New      732
US|...         702
US|***         690
US|Brig        683
India|Shiv     663
India|Pvt      660
dtype: int64


In [ ]:
reference_blocked = reference[
    ["entity_id","business_name","business_address","country","block_key"]
].copy()

source1_blocked = source1[
    ["entity_id","business_name","business_address","country","block_key"]
].copy()

candidates = source1_blocked.merge(
    reference_blocked,
    on="block_key",
    how="inner",
    suffixes=("_s1", "_ref")
)

print("Candidate pairs:", len(candidates))
print("Candidate columns:")
print(candidates.columns)

print("Source 1 records:", len(source1))
print("Candidate pairs:", len(candidates))

print(
    "Average candidates per Source 1 record:",
    len(candidates) / len(source1)
)

In [ ]:
def similarity(name1, name2, address1, address2, country1, country2):

    name_score = ratio(name1, name2)

    address_score = ratio(address1, address2)

    country_score = 100 if country1 == country2 else 0

    final_score = (
        0.55 * name_score +
        0.35 * address_score +
        0.10 * country_score
    )

    return final_score

In [24]:
import time
from pathlib import Path

TRAIN_DIR = Path("../dataset/train")

gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Ground truth shape:", gt.shape)
display(gt.head(254863))

Ground truth shape: (254863, 2)


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
...,...,...
254858,S1-416631146,"S2-249499311,S2-705875694,S3-211053070,S3-5725..."
254859,S1-571678076,"S2-536626853,S3-458159813,S3-181558548"
254860,S1-161181003,"S2-659187720,S3-439954278,S3-899270165,S3-3528..."
254861,S1-529914128,"S2-88880210,S2-412392942,S3-727387978,S3-55137..."


In [21]:
def parse_id_set(value):
    if value is None or value == "":
        return set()

    return {
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    }


gt["true_set"] = gt["matched_entity_ids"].apply(parse_id_set)

display(gt[["source1_entity_id", "matched_entity_ids", "true_set"]].head(10))

,source1_entity_id,matched_entity_ids,true_set
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...","{S2-743505751, S3-860443364, S3-775321672, S2-..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...","{S3-384364074, S2-249013014, S2-197070651, S3-..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467","{S2-479876582, S2-790675320, S3-878454467}"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215","{S3-679606215, S2-24659151, S2-153058913}"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...","{S2-553508714, S3-449308785, S3-928796641, S2-..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762...","{S3-476250621, S3-187831601, S3-641489370, S2-..."
6,S1-318373630,"S2-660036492,S3-804600254","{S3-804600254, S2-660036492}"
7,S1-86989137,"S3-274817120,S3-312496301","{S3-312496301, S3-274817120}"
8,S1-29845983,"S2-648035184,S3-588502663","{S2-648035184, S3-588502663}"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439","{S3-74481402, S3-576451439, S2-383871912}"


In [30]:
print("SOURCE 1 SAMPLE")
display(
    source1[
        ["entity_id", "business_name", "business_address", "country", "exact_key"]
    ].head(5)
)

print("\nREFERENCE SAMPLE")
display(
    reference[
        ["entity_id", "business_name", "business_address", "country", "exact_key"]
    ].head(5)
)

SOURCE 1 SAMPLE


,entity_id,business_name,business_address,country,exact_key
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,"Orelee's Barbershop|1795 Westchester Drive, Hi..."
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,"Prime Money|17560 Ellis Road, Tahlequah, OK|US"
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,"B+ Retail Inc|1712 Montebello Avenue, Phoenix,..."
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,"Christ Chapel|2100 Cameron Drive, Unit APARTME..."
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,"Prabhav Business Center|797, Lake Town Block A..."



REFERENCE SAMPLE


,entity_id,business_name,business_address,country,exact_key
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,राम मार्केटिंग प्राइवेट लिमिटेड|KH NO. -570/13...
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,"-- Holloway Peak Inc Seafood|105 ELM ST, MORGA..."
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,"आदित्य प्रॉपर्टीज एलएलपी|G-3/571, GULMOHAR COL..."
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,"Summit Inc|GREENSBORO, NC, 19 1/2 STARDUST TRA..."
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,"Delta Tetlecommunication Inc|914 PIERPONT AVE,..."


In [39]:
# ============================================================
#CREATE VALIDATION SAMPLE
# ============================================================
print("Creating validation sample...")

TRAIN_DIR = Path("../dataset/train")

gt_full = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Full ground truth:", gt_full.shape)

VALIDATION_SIZE = 5000
RANDOM_SEED = 42

validation_gt = gt_full.sample(
    n=VALIDATION_SIZE,
    random_state=RANDOM_SEED
).copy()

validation_gt = validation_gt.reset_index(drop=True)

print("Validation S1 entities:", len(validation_gt))

needed_s2 = set()
needed_s3 = set()

for value in validation_gt["matched_entity_ids"]:

    if not value:
        continue

    for entity_id in value.split(","):

        entity_id = entity_id.strip()

        if entity_id.startswith("S2-"):
            needed_s2.add(entity_id)

        elif entity_id.startswith("S3-"):
            needed_s3.add(entity_id)

print("Required S2 true-match IDs:", len(needed_s2))
print("Required S3 true-match IDs:", len(needed_s3))

Creating validation sample...
Full ground truth: (254863, 2)
Validation S1 entities: 5000
Required S2 true-match IDs: 8445
Required S3 true-match IDs: 8924


In [2]:
# ============================================================
# SETUP — TRAINING DATA PATH
# ============================================================

from pathlib import Path

TRAIN_DIR = Path("../dataset/train")

print("Training directory:", TRAIN_DIR)
print("Files in training directory:")

for file in TRAIN_DIR.iterdir():
    print(" -", file.name)

Training directory: ../dataset/train
Files in training directory:
 - train_ground_truth.tsv
 - train_source1.tsv
 - train_source3.tsv
 - train_source2.tsv
